# Preprocessing Pipeline (ordered flow)

Reordered and bug-fixed copy of `eeg_data_preprocessing_mne.ipynb` (original preprocessing code by Aurēlija Spunde, based on the MNE-Python tutorials).
All changes are listed in `preprocessing_changes.md`.

**How to use:** run the notebook **once per participant**. Set `SUBJECT` in the parameters cell, inspect the plots, fill in that participant's bad channels and (if needed) ICA components, then run the rest.
Every output is written to a separate `derivatives` folder; the raw files are never modified.

**Flow:** 1. load → 2. events → 3. crop + mark breaks → 4. resample → 5. bad channels → 6. ICA → 7. filter + apply ICA → 8. interpolate → 9. re-reference → 10. save + report

Requirements: `pip install mne mne-icalabel python-picard onnxruntime pandas`

## Loading the data

Code from build.py

In [ ]:
"""

When the real data is available, can run this in the terminal to get the files

uv run build.py --data-root /mnt/FastData/EXPERIMENT_DATA --out out/
uv run build.py --physics        # adds the physics-state table (~17M rows, 266 MB)
uv run build.py --adhoc-session  # a run that is not in sessions.toml (see Curation)
uv run build.py --export-eeg     # also write each aligned hyperscanner file as .fif
"""

Loading the packages

In [ ]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
from mne_icalabel import label_components
%matplotlib inline

## 0. Parameters

All settings live here, so every choice can be reported in the Methods. `TODO` marks things that still need to be checked against the real data.

In [ ]:
# --- Participant to process (run the notebook once per participant) ---
SUBJECT = "1"  # TODO: match the file names produced by build.py --export-eeg

# --- Paths ---
DATA_DIR = Path("/mnt/FastData/EXPERIMENT_DATA")  # TODO: folder with the .fif files from build.py
RAW_FILE = DATA_DIR / f"{SUBJECT}.fif"  # TODO: adapt to the actual file names
OUT_DIR = DATA_DIR / "derivatives" / f"sub-{SUBJECT}"  # outputs stay next to the data, outside the git repo
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Channels ---
MONTAGE = "standard_1020"
EOG_CHANNELS = ["EOG2", "EOG3", "EOG4"]  # TODO: confirm the names (is there an EOG1?)
OTHER_CHANNEL_TYPES = {}  # TODO: add the EMG and IMU channels, e.g. {"EMG1": "emg", "IMU_x": "misc"}

# Bad channels per participant, filled in after inspecting the data in step 5
BAD_CHANNELS = {
    "1": [],
}

# --- Events ---
STIM_CHANNEL = "STI 014"  # only used if the file has no annotations; TODO: check the name
EVENT_ID = {  # adjust to our events (TODO: take the codes from the game repo)
    "individual/balldrop": 1,
    "team/balldrop": 2,
    "visual/left": 3,
    "visual/right": 4,
    "smiley": 5,
    "buttonpress": 32,
}
ROUND_START = "round/start"  # TODO: the event name that marks the start of a round
ROUND_END = "round/end"  # TODO: the event name that marks the end of a round
BREAK_MIN_DURATION = 15.0  # gaps between rounds longer than this (s) are marked as BAD_break

# --- Filtering, resampling and reference ---
SFREQ = 250  # Hz, plenty for alpha and beta, and makes ICA faster
ICA_L_FREQ, ICA_H_FREQ = 1.0, 100.0  # copy used only for ICA (ICLabel expects 1-100 Hz, average reference)
L_FREQ, H_FREQ = 0.1, 40.0  # analysis data
REFERENCE = "average"  # TODO: confirm with the lab; mastoids would be ["M1", "M2"]

# --- ICA ---
ICA_N_COMPONENTS = 0.95  # components explaining 95% of the variance
ICA_REJECT = dict(eeg=200e-6)  # skip 2 s segments with peak-to-peak > 200 µV when fitting ICA
ICLABEL_ARTIFACTS = ["eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"]
ICLABEL_THRESHOLD = 0.8  # exclude a component if ICLabel is at least 80% sure it is an artifact
# After visual review, the final list of components to exclude per participant
# (None = use the automatic selection from step 6)
ICA_EXCLUDE_MANUAL = {
    "1": None,
}
RANDOM_STATE = 97  # fixed seed so ICA gives the same result every run

## 1. Loading the raw data

In [ ]:
# preload=True is needed for filtering, re-referencing and ICA
raw = mne.io.read_raw(RAW_FILE, preload=True)
print(f"Subject: {SUBJECT}, Channels: {raw.info['nchan']}, Duration: {raw.times[-1]:.2f} s, Sampling Rate: {raw.info['sfreq']} Hz")

# If one file contains several participants, keep only this participant's channels here
# TODO: check the structure of the build.py output
# raw.pick([ch for ch in raw.ch_names if ch.startswith(f"P{SUBJECT}_")])

# Channel types: EOG (and EMG/IMU) channels must not be treated as EEG
raw.set_channel_types({ch: "eog" for ch in EOG_CHANNELS if ch in raw.ch_names})
if OTHER_CHANNEL_TYPES:
    raw.set_channel_types(OTHER_CHANNEL_TYPES)

# Set montage (warns instead of failing for channels without a standard position)
raw.set_montage(MONTAGE, on_missing="warn")

# No re-referencing here: that is done at the end, after bad channels are interpolated (step 9)

## 2. Events

Events are stored as **annotations** on the `Raw` object, so they stay correct through cropping, resampling and every later step.
Each participant's own events are used: in the Solo condition the rounds are independent, so the events are not the same for everyone.

In [ ]:
if len(raw.annotations) == 0:
    # No annotations in the file: read the trigger channel and convert the events to annotations
    events = mne.find_events(raw, stim_channel=STIM_CHANNEL)
    annotations = mne.annotations_from_events(
        events,
        sfreq=raw.info["sfreq"],
        event_desc={code: name for name, code in EVENT_ID.items()},
        first_samp=raw.first_samp,
        orig_time=raw.info["meas_date"],
    )
    raw.set_annotations(annotations)

# The trigger channel is no longer needed once the events are annotations
raw.drop_channels([ch for ch, ch_type in zip(raw.ch_names, raw.get_channel_types()) if ch_type == "stim"])

events, event_id = mne.events_from_annotations(raw)
print(event_id)
fig = mne.viz.plot_events(events, event_id=event_id, sfreq=raw.info["sfreq"], first_samp=raw.first_samp)

### Check: do the files of all participants line up?

Run once all files exist. All recordings of a session should have the same start time, sampling rate and length.
The number of events can differ between participants in the Solo condition (independent rounds).

In [ ]:
rows = []
for path in sorted(DATA_DIR.glob("*.fif")):  # TODO: adapt to the actual file names
    r = mne.io.read_raw(path, preload=False, verbose="error")
    row = {"file": path.name, "meas_date": r.info["meas_date"], "sfreq": r.info["sfreq"], "duration_s": round(r.times[-1], 3)}
    if len(r.annotations):
        row.update(pd.Series(r.annotations.description).value_counts().to_dict())
    rows.append(row)
pd.DataFrame(rows)

## 3. Cropping to the task and marking breaks

ICA should only learn from task data. Setup, instructions and long breaks contain movement and talking that do not occur during play.

In [ ]:
def event_times(raw, name):
    """Times (s, relative to the start of the data) of all events with this description."""
    ev, _ = mne.events_from_annotations(raw, event_id={name: 1}, verbose="error")
    return (ev[:, 0] - raw.first_samp) / raw.info["sfreq"]


# Keep only the game: from the first round start to the last round end, with a 2 s margin
starts, ends = event_times(raw, ROUND_START), event_times(raw, ROUND_END)
raw.crop(tmin=max(starts.min() - 2.0, 0), tmax=min(ends.max() + 2.0, raw.times[-1]))

# Mark long gaps between the end of a round and the start of the next one (e.g. the break
# between Solo and Team) as BAD_break, so ICA and later steps ignore them.
# Short gaps are kept, because they may contain feedback events (win/loss).
starts, ends = np.sort(event_times(raw, ROUND_START)), np.sort(event_times(raw, ROUND_END))
break_onsets, break_durations = [], []
for end, next_start in zip(ends[:-1], starts[1:]):
    if next_start - end > BREAK_MIN_DURATION:
        break_onsets.append(end + 1.0)
        break_durations.append(next_start - end - 2.0)  # 1 s margin on each side

# Annotation onsets are counted from the measurement start, so add the time of the first sample
onset_offset = raw.first_time if raw.annotations.orig_time is not None else 0.0
breaks = mne.Annotations(
    onset=np.array(break_onsets) + onset_offset,
    duration=break_durations,
    description="BAD_break",
    orig_time=raw.annotations.orig_time,
)
raw.set_annotations(raw.annotations + breaks)
print(f"Breaks marked: {len(breaks)}, total {np.sum(break_durations):.1f} s of {raw.times[-1]:.1f} s")

## 4. Downsampling to 250 Hz

Done early (on the continuous data, not on epochs) to make every later step faster. The annotations are kept automatically.

In [ ]:
raw.resample(sfreq=SFREQ)

## 5. Identifying bad channels

Bad channels are different for every participant. Inspect the data, then write this participant's bad channels into `BAD_CHANNELS` in the parameters cell.
Tip: with `%matplotlib qt` you can click on channels in `raw.plot()` to mark them as bad.

They are **not** interpolated yet: ICA is fitted without them, and they are interpolated after ICA (step 8).

In [ ]:
raw.plot(n_channels=30, scalings=dict(eeg=50e-6))
raw.compute_psd(fmax=60).plot()

In [ ]:
raw.info["bads"] = BAD_CHANNELS.get(SUBJECT, [])
print("Bad channels:", raw.info["bads"])

## 6. ICA

**Fitting:** ICA is fitted on a copy filtered at 1–100 Hz with average reference (what ICLabel expects), using EEG channels only and skipping the BAD_break segments.
The EOG channels are only used to help identify eye components, not in the decomposition.

**Labeling:** ICLabel classifies each component; EOG correlation is used as a second opinion. Always check the automatic choice visually.

In [ ]:
# Filtering the data for ICA training
raw_ica = raw.copy().filter(l_freq=ICA_L_FREQ, h_freq=ICA_H_FREQ)
raw_ica.set_eeg_reference("average")  # bad channels are left out of the average automatically

ica = ICA(
    n_components=ICA_N_COMPONENTS,
    method="picard",
    fit_params=dict(ortho=False, extended=True),  # extended Infomax, which ICLabel was trained on
    max_iter="auto",
    random_state=RANDOM_STATE,
)
ica.fit(raw_ica, picks="eeg", reject=ICA_REJECT, reject_by_annotation=True)
print(ica)

In [ ]:
# Automatic labeling with ICLabel
ic_labels = label_components(raw_ica, ica, method="iclabel")
components = pd.DataFrame({
    "component": range(ica.n_components_),
    "label": ic_labels["labels"],
    "probability": ic_labels["y_pred_proba"].round(2),
})
is_artifact = components["label"].isin(ICLABEL_ARTIFACTS) & (components["probability"] >= ICLABEL_THRESHOLD)

# Second opinion: components that correlate with the EOG channels
# (computed on the filtered copy, the same data ICA was fitted on)
eog_present = [ch for ch in EOG_CHANNELS if ch in raw_ica.ch_names]
eog_indices = []
if eog_present:
    eog_indices, eog_scores = ica.find_bads_eog(raw_ica, ch_name=eog_present)
components["eog_correlated"] = components["component"].isin(eog_indices)

auto_exclude = sorted(set(components.loc[is_artifact, "component"]) | set(eog_indices))
print("Automatically selected for exclusion:", auto_exclude)
components

In [ ]:
# Visual check of the components
ica.plot_components()
ica.plot_sources(raw_ica, show_scrollbars=False)
if auto_exclude:
    ica.plot_properties(raw_ica, picks=auto_exclude)

In [ ]:
# Final choice: the automatic selection, unless a manual list was set for this participant
manual = ICA_EXCLUDE_MANUAL.get(SUBJECT)
ica.exclude = [int(i) for i in (auto_exclude if manual is None else manual)]
print("Excluded components:", ica.exclude)

# Save the ICA solution and the component table (to report in the Methods)
components["excluded"] = components["component"].isin(ica.exclude)
ica.save(OUT_DIR / f"sub-{SUBJECT}_ica.fif", overwrite=True)
components.to_csv(OUT_DIR / f"sub-{SUBJECT}_ica_components.csv", index=False)

## 7. Filtering the analysis data and applying ICA

The ICA weights from step 6 are applied to the analysis data (0.1–40 Hz).
It is set to average reference first, because ICA was fitted on average-referenced data.
No notch filter is needed: the 40 Hz low-pass already removes the 50 Hz line noise.

In [ ]:
raw_clean = raw.copy().filter(l_freq=L_FREQ, h_freq=H_FREQ)
raw_clean.set_eeg_reference("average")

raw_before_ica = raw_clean.copy()
ica.apply(raw_clean)
ica.plot_overlay(raw_before_ica, picks="eeg")  # before vs. after removing the components

## 8. Interpolating bad channels

In [ ]:
bad_channels = list(raw_clean.info["bads"])  # keep the list for the summary
raw_clean.interpolate_bads(reset_bads=True)

## 9. Re-referencing

Done after interpolation, so the interpolated channels are part of the reference.

In [ ]:
raw_clean.set_eeg_reference(REFERENCE)
raw_clean.plot(n_channels=30, scalings=dict(eeg=50e-6))

## 10. Saving the cleaned data and a quality report

The report (HTML) shows the data before and after cleaning and the ICA components. The summary table collects the numbers to report in the Methods for all participants.

In [ ]:
raw_clean.save(OUT_DIR / f"sub-{SUBJECT}_clean_raw.fif", overwrite=True)

report = mne.Report(title=f"Preprocessing sub-{SUBJECT}")
report.add_raw(raw, title="Before cleaning (cropped, resampled)", psd=True)
report.add_ica(ica, title="ICA", inst=raw_ica)
report.add_raw(raw_clean, title="After cleaning", psd=True)
report.save(OUT_DIR / f"sub-{SUBJECT}_report.html", overwrite=True, open_browser=False)

# One row per participant; re-running a participant replaces its row
summary_file = OUT_DIR.parent / "preprocessing_summary.csv"
row = pd.DataFrame([{
    "subject": SUBJECT,
    "bad_channels": " ".join(bad_channels),
    "n_bad_channels": len(bad_channels),
    "n_ica_components": ica.n_components_,
    "excluded_components": " ".join(map(str, ica.exclude)),
    "n_excluded_components": len(ica.exclude),
    "break_s": round(float(np.sum(break_durations)), 1),
    "duration_s": round(raw_clean.times[-1], 1),
}])
if summary_file.exists():
    previous = pd.read_csv(summary_file, dtype={"subject": str})
    row = pd.concat([previous[previous["subject"] != SUBJECT], row])
row.to_csv(summary_file, index=False)
row

## Next: analysis (separate notebook)

The ERP epoching from the original notebook (−0.2 to 0.5 s around events) was removed, because it does not fit the cRQA analysis. The next steps work on the cleaned continuous data:

1. Band-pass alpha (8–12 Hz) and beta (13–30 Hz) and compute the Hilbert amplitude envelope on the **continuous** data (cutting first would add filter edge artifacts to every segment).
2. Cut the envelopes into rounds using the round start/end events.
3. Average over ROIs, downsample the envelopes to ~20–50 Hz, then run cRQA.